# Practice 31 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — `.loc` vs `.iloc`

### Task 1: tips — when labels and positions diverge

Load `sns.load_dataset('tips')`.

1. Sort the DataFrame by `total_bill` descending. Then:
   - What does `df.loc[0]` return? What does `df.iloc[0]` return? Are they the same row? Explain why.
   - What is the highest total_bill in the dataset, and who paid it? Use `.iloc`.

2. Filter to only Female diners using `.loc` with a boolean mask. On the filtered result:
   - Print the index — what do you notice?
   - Use `.iloc[0]` to get the first female diner. Use `.loc[0]` — do you get the same row? Why or why not?

3. Set `'day'` as the index. Then:
   - Use `.loc['Fri', 'total_bill']` to get all Friday total bills.
   - Use `.iloc[0, 1]` to get the first row's tip by position.
   - Use `.loc['Sat', ['tip', 'total_bill']]` to get two columns for Saturday.

In [29]:
# Your code here

tips = sns.load_dataset('tips')


tips = tips.sort_values(by = 'total_bill',ascending=False)

tips.loc[0]
### it returns the first row of the original tips 
tips.iloc[0]
### it returns the first row of the sorted df 

tips.iloc[0]['total_bill']


f = tips.loc[tips['sex']=='Female']
print(f.index)
f.loc[0]

f.iloc[0]
## don't get the same row , loc is on the original unsorted tips index, .iloc is on the current first row 


tips = tips.set_index('day')

tips
ft = tips.loc['Fri','total_bill']
tips.iloc[0,1]
tips.loc['Sun',['tip','total_bill']]

Index([102, 197, 238,  11,  85,  52, 219, 155, 125, 214, 240, 143,  72,  57,
       114,  73, 157,   4, 119,  94, 103, 229, 104, 186,  33,  21, 131, 191,
        29, 243, 146, 134, 188, 164, 140, 115,  71,   0,  18,  37, 205,  66,
       144, 203,  93, 225, 162, 223,  22, 101,  32,  14,  74, 127, 109, 137,
       121, 221, 158, 213, 139, 198, 202, 215, 209, 201, 124, 118, 133, 147,
       128, 100, 132, 117, 169, 168,  16, 136,  51, 226,  82, 178, 135, 145,
       111,  92,  67],
      dtype='int64')


,tip,total_bill
day,,
Sun,5.00,48.17
Sun,3.50,45.35
Sun,3.00,40.55
Sun,4.00,38.07
Sun,5.00,35.26
...,...,...
Sun,1.56,9.94
Sun,1.32,9.68
Sun,4.00,9.60


---
## Level 2 — Category + `.loc`

### Task 2: mpg — filtering with ordered categories

Load `sns.load_dataset('mpg')`. Convert `cylinders` to an ordered category (`3 < 4 < 5 < 6 < 8`) and `origin` to unordered category.

Answer these questions — use `.loc` for all filtering:

1. Filter to cars with more than 4 cylinders. What fraction are from the USA? (One `.loc` call — select rows and the `origin` column together.)
2. Filter to USA 6-cylinder cars. Which `model_year` had the best average `mpg` among them?
3. Add a `weight_class` column using `pd.cut` with 3 bins of your choice. Use `.loc` to extract only the heaviest class — what is the mean `mpg` for those cars?
4. Among cars with `cylinders` strictly less than 6, which `origin` has the highest median `mpg`? Use `observed=True`.

In [52]:
# Your code here

mpg = sns.load_dataset('mpg')

co = pd.CategoricalDtype(categories=[3,4,5,6,8],ordered=True)
mpg['cylinders'] = mpg['cylinders'].astype(co)

f4 = mpg.loc[mpg['cylinders']>4]
np.mean(f4['origin']=='usa')

f6 = mpg.loc[(mpg['cylinders']==6) & (mpg['origin']=='usa')]

f6.groupby('model_year')['mpg'].mean().idxmax()

mpg['weight_class'] = pd.cut(mpg['weight'],3,labels=['light','mid','heavy'])

fh = mpg.loc[mpg['weight_class']=='heavy']
np.nanmean(fh['mpg'])

f6 = mpg.loc[mpg['cylinders']<6]
f6.groupby('origin', observed=True)['mpg'].median().idxmax()

'japan'

---
## Level 3 — Aggregation

### Task 3: planets — group and reshape

Load `sns.load_dataset('planets')`. No `.pipe()` in this task.

- Convert `method` to `category`. Drop rows where `orbital_period` or `mass` is null.
- For each detection method, compute: total planets discovered, mean distance, median orbital_period. Produce this as a single DataFrame using `.agg()` with `observed=True`.
- Use `.loc` on the result to extract rows for methods that discovered more than 100 planets total.
- Build a pivot table: mean `mass` by `method` × `year` (treat `year` as-is, no category conversion). Use `.xs()` to extract one method's row of your choice.
- Use a list comprehension to find `(method, year)` pairs in the pivot table where mass data exists (is not NaN). How many are there?

In [89]:
# Your code here

planets = sns.load_dataset('planets')
print(planets.shape)

planets['method'] = planets['method'].astype('category')

planets.dropna(subset=['orbital_period','mass'],inplace=True)
print(planets.shape)

t = planets.groupby('method',observed=True)[['number','distance','orbital_period']].agg(['sum','mean','median'])


t.loc[t['number']['sum']>100]

p = planets.pivot_table(
    index='method',
    columns='year',
    values= 'mass',
    observed=True
)
p.xs('Transit')


#Use a list comprehension to find `(method, year)` pairs in the 
# pivot table where mass data exists (is not NaN). How many are there?
ps = p.stack(dropna=False)

l = [i for i in ps.index if not pd.isna(ps[i])]
len(l)

(1035, 6)
(513, 6)


/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_49265/4208912186.py:27: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  ps = p.stack(dropna=False)


24

---
## Level 4 — Pipe

### Task 4: titanic — pipeline and survival analysis

Load `sns.load_dataset('titanic')`. Write a `.pipe()` chain with 3 functions:

- One applies ordered `CategoricalDtype` to `pclass` (`3 < 2 < 1`) and converts `sex` to `category`
- One adds a `fare_per_class` column — mean fare within each pclass using `transform`
- One adds a boolean `above_avg_fare` column (True if the passenger paid above their class average)

After the chain:
- Use `.loc` with a boolean mask to get passengers who are both above average fare **and** survived. What fraction of all survivors does this represent?
- Among passengers with `pclass > 3` (i.e. class 1 and 2), what is the survival rate for `above_avg_fare=True` vs `False`? Use `np.mean()`.
- Build a pivot table: mean `above_avg_fare` (as float) by `pclass` × `sex`, `observed=True`. Which group has the highest rate of above-average fare payers?

In [88]:
# Your code here
titanic = sns.load_dataset('titanic')
print(titanic.shape)
po = pd.CategoricalDtype(categories=[3,2,1],ordered=True)

def pclass_order(df):
    df['pclass'] = df['pclass'].astype(po)
    df['sex'] = df['sex'].astype('category')
    return df 

def fpc(df):
    df['fare_per_class'] = df.groupby('pclass',observed = True)['fare'].transform('mean')
    return df 

def aaf(df):
    df['above_avg_fare'] = df['fare']>df['fare_per_class']
    return df 


titanic.pipe(pclass_order).pipe(fpc).pipe(aaf)
print(titanic.shape)


f = titanic.loc[(titanic['above_avg_fare'])&(titanic['survived'] == 1)]
f['survived'].sum()/titanic['survived'].sum()

titanic.loc[titanic['pclass']>3].groupby('above_avg_fare')['survived'].mean()

p = titanic.pivot_table(
    index = 'pclass',
    columns='sex',
    values='above_avg_fare',
    observed=True
)
p.stack().idxmax()

(891, 15)
(891, 17)


(np.int64(2), 'female')